# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SodiqAbdulwaris/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule:** a page is worth reviewing first if it hasn't been touched in 180+ days (**stale**) AND is still pulling real search traffic — 500+ impressions in the last 90 days (**visible**). Among pages that pass both, rank by how much exposure they have: the more visible a stale page is, the more an editor's hour is worth spending on it. Same threshold as the Week 2 demo rule, kept frozen here — see Section 4 for why loosening it turns out to be a trap, not an improvement.

**Reason codes:**
- `stale_but_visible` — passed both core conditions (the only code the score itself can produce)
- `thin_content` — under 1,200 words; an editorial flag independent of the score
- `page_one_low_ctr` — top-10 average position but CTR below the scored group's median (an opportunity, not decline)
- `general_review` — passed neither core condition; in the queue only because everything gets ranked, not because the rule found anything

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
# label used only to EVALUATE the rule below -- never an input to the score or reason codes
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["baseline_score"] = stale * visible * df["impressions_90d"]
print("pages passing stale AND visible:", (df['baseline_score'] > 0).sum(), "/", len(df))


pages passing stale AND visible: 17 / 30000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Score, reason codes, rank -- then write the full queue to `work/outputs/baseline_action_score.csv` (gitignored working artifact) and a small metrics summary to `work/outputs/baseline_metrics.json` (committed -- the receipt these numbers trace back to).

In [2]:
import json
from pathlib import Path
import sys
sys.path.insert(0, "scripts")
from ml_utils import precision_at_k
from sklearn.model_selection import GroupShuffleSplit

# CTR threshold computed only from pages the rule already scored -- not from the full population
ctr_median = df.loc[df["baseline_score"] > 0, "ctr"].median()

def reason_codes(row):
    reasons = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_but_visible")
    if row["word_count"] > 0 and row["word_count"] < 1200:
        reasons.append("thin_content")
    if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["ctr"] < ctr_median:
        reasons.append("page_one_low_ctr")
    if not reasons:
        reasons.append("general_review")
    return "|".join(reasons)

df["reason_codes"] = df.apply(reason_codes, axis=1)
df["baseline_rank"] = df["baseline_score"].rank(method="first", ascending=False).astype(int)

queue_cols = ["content_id", "client_id", "baseline_rank", "baseline_score", "reason_codes",
              "impressions_90d", "avg_position", "ctr", "word_count", "days_since_last_update",
              "is_declining_label", "trend_direction"]
queue = df.sort_values("baseline_rank")[queue_cols]

Path("work/outputs").mkdir(parents=True, exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("wrote", len(queue), "rows to work/outputs/baseline_action_score.csv")

# honest evaluation: client-holdout, same split style the model notebooks use later
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
_, test_idx = next(gss.split(df, groups=df["client_id"]))
test = df.iloc[test_idx]
base_rate_test = float(test["is_declining_label"].mean())

metrics = {
    "rows_scored": int(len(df)),
    "rows_passing_rule": int((df["baseline_score"] > 0).sum()),
    "test_clients": int(test["client_id"].nunique()),
    "test_pages": int(len(test)),
    "base_rate_test": round(base_rate_test, 3),
    "precision_at_20_client_holdout": round(precision_at_k(test["is_declining_label"], test["baseline_score"], 20), 3),
    "precision_at_50_client_holdout": round(precision_at_k(test["is_declining_label"], test["baseline_score"], 50), 3),
}
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))


wrote 30000 rows to work/outputs/baseline_action_score.csv
{
  "rows_scored": 30000,
  "rows_passing_rule": 17,
  "test_clients": 8,
  "test_pages": 7115,
  "base_rate_test": 0.517,
  "precision_at_20_client_holdout": 0.589,
  "precision_at_50_client_holdout": 0.545
}


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Action, confidence, and "what would make it wrong" derived programmatically from each row's reason codes and score -- then read the actual top 20 by hand below the table.

In [3]:
def action_for(reasons):
    if "thin_content" in reasons:
        return "expand_and_refresh"
    if "page_one_low_ctr" in reasons:
        return "refresh_and_review_ctr"
    if "stale_but_visible" in reasons:
        return "refresh"
    return "monitor_only"

def confidence_for(row):
    if row["baseline_score"] == 0:
        return "none -- zero-score tie, not a real recommendation"
    if "|" in row["reason_codes"]:
        return "high -- multiple independent signals agree"
    return "medium -- single clear driver (staleness x visible traffic)"

def wrong_if_for(row):
    if row["baseline_score"] == 0:
        return "always -- rank is arbitrary CSV row order, the rule flagged nothing here"
    if row["is_declining_label"] == 0:
        return "already wrong here: stale + visible, but trend_direction is not 'down'"
    return "wrong if traffic held steady after the snapshot, or a refresh is already in flight"

top20 = queue.head(20).copy()
top20["action"] = top20["reason_codes"].apply(action_for)
top20["confidence"] = top20.apply(confidence_for, axis=1)
top20["would_be_wrong_if"] = top20.apply(wrong_if_for, axis=1)

review_cols = ["baseline_rank", "baseline_score", "reason_codes", "action", "confidence",
               "is_declining_label", "trend_direction", "would_be_wrong_if"]
print(top20[review_cols].to_string(index=False))


 baseline_rank  baseline_score      reason_codes       action                                                  confidence  is_declining_label trend_direction                                                                  would_be_wrong_if
             1           61678 stale_but_visible      refresh medium -- single clear driver (staleness x visible traffic)                   1            down wrong if traffic held steady after the snapshot, or a refresh is already in flight
             2           59472 stale_but_visible      refresh medium -- single clear driver (staleness x visible traffic)                   1            down wrong if traffic held steady after the snapshot, or a refresh is already in flight
             3           25715 stale_but_visible      refresh medium -- single clear driver (staleness x visible traffic)                   1            down wrong if traffic held steady after the snapshot, or a refresh is already in flight
             4           13299 stale

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak / wrong picks:**
- **Rank 12** is flagged `stale_but_visible` (stale, 500+ impressions) but its `trend_direction` is `stable`, not `down` -- a real miss. It's old and still gets traffic, but wasn't actually losing ground. A rule built only from staleness and visibility has no way to see an ongoing trend; only trend-derived fields could tell it apart, and those are excluded as leakage (ML-04). That's the honest cost of a rule built entirely from pre-decision signals.
- **Only 17 of 30,000 pages (0.06%) ever pass `stale AND visible`.** 99.4% of this slice was updated within 180 days (median: 20 days) -- the 180-day threshold barely fires here. Every rank past 17 is a **zero-score tie**, ordered by CSV row order for display, not by the rule. On the held-out clients only 3 pages receive a positive score, so 47 of the top-50 slots are tied at zero. `precision_at_k` now reports the expected hit rate across that tied group instead of letting row order or pandas choose the metric: 0.545 at K=50 versus a 0.517 base rate.
- **I tried loosening `stale` to `>= 90` days as a fix -- it backfires.** 9,345 rows pass, but of the 6,575 that also clear `visible`, the vast majority share an *exact* `days_since_last_update` value of **104** (8,773 rows overall, 29.2% of the whole slice) -- a suspiciously common single value `docs/data-dictionary.md` doesn't explain, almost certainly a capped/censored measurement rather than 8,773 pages independently last touched on day 104. Ranking those by raw `impressions_90d` doesn't help either: `corr(impressions_90d, is_declining_label)` across the whole slice is **-0.018**, essentially zero -- "biggest audience among the stale ones" isn't a decline signal here. Client-holdout precision at that looser threshold actually falls *below* the 0.517 base rate (0.20 @20, 0.30 @50) -- worse than guessing. The narrow, precedented 180-day rule stays: precise when it fires (16/17 correct), honestly limited past rank 17, and clearly beatable -- exactly what a baseline should be.

**Leakage check:** the score (`stale`, `visible`, `impressions_90d`) and every reason code (`days_since_last_update`, `word_count`, `avg_position`, `ctr`) come only from the **Feature** bucket of the ML-04 contract. `trend_direction` / `trend_pct` -- the label's source -- never appear in the score, the reason codes, or the ranking; they're read only afterward, to *evaluate* the queue against `is_declining_label`, never to build it. No product-decision flags (`provider_used`, `model_used`) or private identifiers beyond `content_id`/`client_id` (grouping and holdout only) are used anywhere.

In [4]:
# Confirms the two claims above with code rather than eyeballing them

loose_stale = (df["days_since_last_update"] >= 90).astype(int)
loose_score = loose_stale * visible * df["impressions_90d"]
print("loosened rule (>=90d): rows passing:", (loose_score > 0).sum())
print("of those, share with days_since_last_update == 104 exactly:",
      round((df.loc[loose_score > 0, "days_since_last_update"] == 104).mean(), 3))
print("days_since_last_update == 104, whole slice:", (df['days_since_last_update'] == 104).sum(),
      f"({(df['days_since_last_update'] == 104).mean():.1%})")

print("\ncorr(impressions_90d, is_declining_label):", round(df["impressions_90d"].corr(df["is_declining_label"]), 3))

gss_loose = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
_, loose_test_idx = next(gss_loose.split(df, groups=df["client_id"]))
loose_test = df.iloc[loose_test_idx]
loose_test_score = loose_test["days_since_last_update"].ge(90).astype(int) * loose_test["impressions_90d"].ge(500).astype(int) * loose_test["impressions_90d"]
print("\nloosened rule, client-holdout base rate:", round(loose_test["is_declining_label"].mean(), 3))
for k in (20, 50):
    print(f"loosened rule Precision@{k}:", round(precision_at_k(loose_test["is_declining_label"], loose_test_score, k), 3))

# leakage check: score/reason-code inputs vs the label-source columns
score_and_reason_inputs = {"days_since_last_update", "impressions_90d", "word_count", "avg_position", "ctr"}
label_source_cols = {"trend_direction", "trend_pct"}
print("\nscore/reason-code inputs overlap label-source columns (should be empty):",
      sorted(score_and_reason_inputs & label_source_cols))


loosened rule (>=90d): rows passing: 6575
of those, share with days_since_last_update == 104 exactly: 0.981
days_since_last_update == 104, whole slice: 8773 (29.2%)

corr(impressions_90d, is_declining_label): -0.018

loosened rule, client-holdout base rate: 0.517
loosened rule Precision@20: 0.2
loosened rule Precision@50: 0.3

score/reason-code inputs overlap label-source columns (should be empty): []


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.